# NB2 · Preparing the data

**Building Clinical Decision Support Systems with Generative AI Tools**  
Technology and Artificial Intelligence Literacy Training in Health Sciences · Akdeniz University · 18 September 2026

Prof. Dr. Utku Köse · Süleyman Demirel University, Department of Computer Engineering  
Director, Artificial Intelligence Application and Research Center (YAZEM) · utkukose@sdu.edu.tr

---


## How this notebook works

You will not write the code. Each step provides a prompt. Copy the prompt into a
generative AI tool, take the code it produces, paste that code into the blank cell
below the prompt and run it.

Every paste cell is followed by a check cell. The check cell tests whether the code you
produced returns what was asked for, and reports anything missing. The check cells are
the fixed part of this notebook and should not be modified.

Each prompt ends with a section headed CONTRACT. The contract states which variables
the code must produce and under which names, and the check cell tests exactly that.
When a task is delegated to a generative AI tool, defining the interface is the user's
responsibility. A prompt without a contract produces code that cannot be checked.

When a check reports a shortcoming, return the generated code to the AI tool, state
what was reported and have the code regenerated. This loop is normal; a contract is
rarely satisfied on the first attempt.


## The problem addressed here

The task is to predict whether a patient admitted to intensive care will stay longer
than three days. The clinical counterpart of this question is bed capacity planning. A
patient identified in advance as a long stay allows capacity to be arranged around
them; when no such identification is made, capacity fills without a plan and the
admission of the next patient is delayed.

The prediction is made six hours after the patient enters intensive care. That point is
the decision moment of the system. Data recorded up to the decision moment may be used,
and data recorded after it may not. This rule is the central rule of the session and
its rationale is set out in step three.

The data source is the MIMIC-IV demo dataset. It covers one hundred patients, is openly
accessible and requires no credentialing. The files are read directly from the web,
with no download step.


## Setup


In [ ]:
!pip -q install pandas numpy

import urllib.request

REPO = 'https://raw.githubusercontent.com/utkukose/cdss-genai-NB-lecture/main'
urllib.request.urlretrieve(f'{REPO}/workshop/checks.py', 'checks.py')

import pandas as pd
import checks
checks.LANG = 'en'

print('Check module ready.')


---

## Step 1 · Fetch the tables

MIMIC data is held in separate linked tables. The age and sex of the patient sit in one
table, hospital admission details in a second, and intensive care length of stay in a
third. This step fetches all three.

The prompt states the file addresses and the column names explicitly. A generative AI
tool should not be expected to know the schema. Where the schema is withheld, the tool
fills the gap by guessing, and the guess is usually wrong. Supplying the data
dictionary is the user's responsibility.


### Prompt 1

```
Write a single Python cell that runs in Google Colab.

Read the following three files directly from the web with pandas. All three are
gzipped CSV files:

https://physionet.org/files/mimic-iv-demo/2.2/hosp/patients.csv.gz
https://physionet.org/files/mimic-iv-demo/2.2/hosp/admissions.csv.gz
https://physionet.org/files/mimic-iv-demo/2.2/icu/icustays.csv.gz

The column structure is:
patients:   subject_id, gender, anchor_age
admissions: subject_id, hadm_id, admission_type, insurance, race
icustays:   subject_id, hadm_id, stay_id, first_careunit, intime, outtime, los

intime and outtime are timestamps; convert them to pandas datetime.
los gives the intensive care length of stay in days.

CONTRACT
When the code completes, these three variables must exist:
  patients_tbl   -> the patients table
  admissions_tbl -> the admissions table
  icustays_tbl   -> the icustays table
Print the row count of each table. Produce no other output.
```


In [ ]:
# Paste the generated code into this cell and run it.


### Check 1


In [ ]:
checks.check_tables(patients_tbl=patients_tbl,
                    admissions_tbl=admissions_tbl,
                    icustays_tbl=icustays_tbl)


---

## Step 2 · Remove stays that never reach the decision moment

Some intensive care stays end before six hours have passed. Those patients never reach
the decision moment of the system, so producing a prediction about them is meaningless.
Such stays must be removed from the working set.

This removal looks like a minor operation, but it is part of the cohort definition.
Which patients a clinical decision support system covers matters as much as which
patients it leaves outside its scope.


### Prompt 2

```
Build the working set from icustays_tbl. Write a single Python cell.

Remove rows where los is below 0.25 days. Those patients left intensive care before
the six hour mark.

Print how many rows were removed and how many remain.

CONTRACT
The result must be a DataFrame named df.
df must contain subject_id, hadm_id, stay_id, first_careunit, intime, outtime
and los.
```


In [ ]:
# Paste the generated code into this cell and run it.


### Check 2


In [ ]:
checks.check_columns(
    df,
    required=['subject_id', 'hadm_id', 'stay_id', 'first_careunit',
              'intime', 'outtime', 'los'],
)

rows_after_step2 = len(df)
print(f'\nRows at the end of step 2: {rows_after_step2}')


---

## Step 3 · Create the target variable

The target is derived from the los column. Stays longer than three days take the value
1 and all others take 0.

One point requires attention here. The value of los becomes known only after the
patient leaves intensive care. If the column is given to the model as an input, the
model shows near perfect performance. Once the system is deployed in a hospital,
however, that column is empty at the decision moment and no output can be produced.

This situation is called target leakage. It is the most common error in clinical code
written with generative AI. It proceeds silently, raises no error, and appears
favourable at first sight because it raises the performance figures.

The same reasoning applies to outtime. A model that knows the discharge time also knows
the length of stay.

Both columns are therefore removed once the target has been created. Leakage is
prevented at the prompt stage rather than discovered afterwards.


### Prompt 3

```
Create the target variable in df and remove the columns that cause leakage. Write a
single Python cell.

1. Create a new column named target. It takes the value 1 where los is greater than
   3, and 0 otherwise.
2. After the target has been created, drop the los and outtime columns from df.

Print the distribution of the target variable.

CONTRACT
df must contain a column named target taking only the values 0 and 1.
df must NOT contain los or outtime.
The row count of df must not change.
```


In [ ]:
# Paste the generated code into this cell and run it.


### Check 3

Three separate checks run at this step: The structure of the target variable, the state
of the columns that had to be removed, and whether the row count was preserved.


In [ ]:
checks.check_target(df, target='target')


In [ ]:
checks.check_columns(df, required=['target'], forbidden=['los', 'outtime'])


In [ ]:
checks.check_rows(df, before=rows_after_step2)


---

## Step 4 · Add demographic and admission information

So far the working set holds only information from the intensive care table. Age, sex
and admission type sit in the other two tables, and all of them are present in the
patient record at the decision moment.

One failure mode in a merge goes unnoticed. Where the join key is chosen incorrectly,
rows multiply and the same stay appears in the table more than once. No error is
raised; only the row count grows. The row count is therefore checked separately after
the merge.


### Prompt 4

```
Add patient and admission information to df. Write a single Python cell.

1. Add gender and anchor_age from patients_tbl, joined on subject_id.
2. Add admission_type and insurance from admissions_tbl, joined on hadm_id.

Use left joins. The row count of df must not change.

Print the row count before and after the merge.

CONTRACT
df must contain these columns:
  subject_id, stay_id, target, gender, anchor_age, admission_type,
  insurance, first_careunit
The row count of df must equal the row count from the previous step.
```


In [ ]:
# Paste the generated code into this cell and run it.


### Check 4


In [ ]:
checks.check_columns(
    df,
    required=['subject_id', 'stay_id', 'target', 'gender', 'anchor_age',
              'admission_type', 'insurance', 'first_careunit'],
    forbidden=['los', 'outtime'],
)


In [ ]:
checks.check_rows(df, before=rows_after_step2)


If the row count has grown, the join key is wrong. A patient may have more than one
hospital admission, so subject_id alone is not a key at stay level. Correct the prompt
and have the code regenerated.


---

## Step 5 · Check the cohort as a whole

The steps have been checked one by one. The cohort is now tested as a whole. This check
repeats the earlier tests and adds two further inspections.

The first concerns the patient identifier. A patient may have more than one intensive
care stay. Where that is the case, the split into training and test sets must be made
at patient level. If one stay of a patient falls into training and another into test,
the model recognises that patient and test performance comes out higher than it should.
The matter is addressed in NB3.

The second is a leak scan based on column names. The scan inspects names only and
offers no guarantee. The substantive question belongs to the user: Was this column
populated at the decision moment?


In [ ]:
checks.check_cohort(df, target='target', patient_id='subject_id')


---

## Step 6 · Other types of data

Routine hospital data has been used so far. That data type is a table of numeric and
categorical columns. Some participants work instead on images, physiological time
series or clinical text.

The prompts below carry out the same task for different data types. Select the one that
matches your own field. Every step after this point proceeds identically regardless of
data type.

Where no real data is available, the prompts generate synthetic data. What is learned
from synthetic data is the method itself, and teaching the method is the purpose of
this session.


### Prompt 6a · Medical imaging

```
Write a single Python cell that loads a suitable subset of MedMNIST for the problem.
If no suitable collection exists, generate a synthetic image set with a controllable
signal.

My problem: [write your own problem here in one sentence]

Print the class distribution. Hold out the test set before any preprocessing step is
fitted. Display a four by four grid of examples.

CONTRACT
The variables X_train, X_test, y_train and y_test must exist.
The values of y must be only 0 and 1.
```


### Prompt 6b · Physiological time series

```
Write a single Python cell that prepares physiological time series data. Use the
MIMIC-IV-ECG demo set if it fits, otherwise generate synthetic signals.

My problem: [write your own problem here in one sentence]

Print the sampling rate and the window length. Split training and test at patient
level rather than window level, and add the reason as a comment.

CONTRACT
The variables X_train, X_test, y_train and y_test must exist.
The variables patient_id_train and patient_id_test must also exist.
```


### Prompt 6c · Clinical text

```
Write a single Python cell that generates a synthetic corpus of clinical notes.

My problem: [write your own problem here in one sentence]
The notes must be in English.

Include the documentation features that make real clinical text difficult:
abbreviations, negation, expressions of uncertainty, templated boilerplate and copy
forward from earlier notes. The label must not be recoverable from a single keyword;
state which surface cues you deliberately removed.

CONTRACT
A DataFrame named df must exist, containing text and target columns.
A subject_id column must also be present.
Print three example notes with their labels.
```


### Prompt 6d · Your own tabular data

```
Write a single Python cell that generates a synthetic patient table for my problem.

My problem: [write here]
The outcome I want to predict: [write here]
The frequency of that outcome in my own patient group: [write as a percentage]
The information available to me at the decision moment: [list it]

The outcome rate must be the percentage I stated. Add missing data at a realistic
rate. Include no column that would be recorded only after the outcome occurs.

CONTRACT
A DataFrame named df must exist, containing target and subject_id columns.
Add a comment at the top of the cell stating that this is synthetic data generated
for teaching.
```


In [ ]:
# Paste the code you generated for your own data type into this cell.


### Check 6

Participants continuing with tabular data run the cell below unchanged. Those working
with images or time series may skip this check; their contract concerns the X and y
variables instead.


In [ ]:
checks.check_cohort(df, target='target', patient_id='subject_id')


---

## End of notebook · Data report

The cell below is the fixed evaluation section of this notebook. It runs on whatever
table you produced and reports what needs to be known before moving to NB3.


In [ ]:
checks.data_report(df, target='target', patient_id='subject_id')


The report is expected to raise a warning about the number of positive cases. The
results of a model built on a cohort of one hundred patients demonstrate the method
rather than establishing performance. The numerical form of that distinction is
addressed in NB3.


## What this notebook covered

A working data pipeline was built without writing code. Three practices were adopted
along the way.

The data dictionary was placed in the prompt. The generative AI tool was not expected
to know the schema; table names, column names and file addresses were stated
explicitly.

A contract was attached to every prompt. This made it possible to test whether the
generated code returned what was asked for, and the testing was carried out step by
step.

The decision moment was defined and columns known only afterwards were removed. Leakage
was prevented at the prompt stage rather than discovered later.

The cost of producing code has fallen; responsibility for determining which code to ask for
remains with the user.

---

**Warning.** Nothing produced in this notebook is a validated clinical tool. The
MIMIC-IV demo data comes from a single hospital in the United States and does not
represent an intensive care population elsewhere. The material is for teaching.
